In [ ]:
import zipfile
import os
import json
import glob

zip_path = "/content/odis_male_json.zip"
extract_path = "/content/odi_data"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

json_files = glob.glob(extract_path + "/**/*.json", recursive=True)

print("Total ODI JSON files:", len(json_files))

Total ODI JSON files: 2571


In [ ]:
with open(json_files[0], "r", encoding="utf-8") as f:
    match = json.load(f)

print("Top-level keys:", match.keys())

info = match["info"]

print("Teams:", info.get("teams"))
print("Date:", info.get("dates"))
print("Venue:", info.get("venue"))
print("Match Type:", info.get("match_type"))
print("Gender:", info.get("gender"))
print("Outcome:", info.get("outcome"))

Top-level keys: dict_keys(['meta', 'info', 'innings'])
Teams: ['Pakistan', 'West Indies']
Date: ['2016-10-05']
Venue: Sheikh Zayed Stadium
Match Type: ODI
Gender: male
Outcome: {'by': {'runs': 136}, 'winner': 'Pakistan'}


In [ ]:
import pandas as pd

matches = []

for file in json_files:
    try:
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)

        info = data["info"]
        teams = info.get("teams", [])
        outcome = info.get("outcome", {})

        if len(teams) != 2:
            continue

        if "winner" not in outcome:
            continue

        winner = outcome["winner"]

        if winner not in teams:
            continue

        matches.append({
            "match_id": os.path.splitext(os.path.basename(file))[0],
            "date": str(info.get("dates", [""])[0]),
            "season": info.get("season"),
            "venue": info.get("venue", "Unknown"),
            "city": info.get("city", "Unknown"),
            "team_1": teams[0],
            "team_2": teams[1],
            "winner": winner,
            "match_type": info.get("match_type"),
            "gender": info.get("gender"),
            "event": info.get("event", {}).get("name", "Unknown"),
            "toss_winner": info.get("toss", {}).get("winner", "Unknown"),
            "toss_decision": info.get("toss", {}).get("decision", "Unknown")
        })

    except Exception:
        continue

odi_matches = pd.DataFrame(matches)

odi_matches["date"] = pd.to_datetime(
    odi_matches["date"],
    errors="coerce"
)

odi_matches = (
    odi_matches
    .sort_values("date")
    .reset_index(drop=True)
)

print("Valid ODI matches:", len(odi_matches))
odi_matches.head()

Valid ODI matches: 2442


,match_id,date,season,venue,city,team_1,team_2,winner,match_type,gender,event,toss_winner,toss_decision
0,66283,2002-06-27,2002,"Trent Bridge, Nottingham",Nottingham,England,Sri Lanka,England,ODI,male,NatWest Series,England,bat
1,66284,2002-06-29,2002,"Lord's, London",London,England,India,India,ODI,male,NatWest Series,England,bat
2,64814,2002-12-29,2002/03,"McLean Park, Napier",Napier,New Zealand,India,New Zealand,ODI,male,India tour of New Zealand,India,field
3,64815,2003-01-01,2002/03,"Jade Stadium, Christchurch",Christchurch,India,New Zealand,New Zealand,ODI,male,India tour of New Zealand,India,bat
4,64816,2003-01-04,2002/03,"Davies Park, Queenstown",Queenstown,India,New Zealand,New Zealand,ODI,male,India tour of New Zealand,New Zealand,field


In [ ]:
team_performance = []

for file in json_files:
    try:
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)

        info = data["info"]
        teams = info.get("teams", [])

        if len(teams) != 2:
            continue

        match_id = os.path.splitext(os.path.basename(file))[0]

        for innings in data.get("innings", []):
            team = innings.get("team")

            if team not in teams:
                continue

            total_runs = 0
            wickets = 0
            legal_balls = 0

            for over in innings.get("overs", []):
                for delivery in over.get("deliveries", []):

                    runs = delivery.get("runs", {})
                    total_runs += runs.get("total", 0)

                    extras = delivery.get("extras", {})

                    if "wides" not in extras and "noballs" not in extras:
                        legal_balls += 1

                    for wicket in delivery.get("wickets", []):
                        dismissal = wicket.get("kind", "")

                        if dismissal not in [
                            "retired hurt",
                            "retired not out",
                            "obstructing the field"
                        ]:
                            wickets += 1

            overs = legal_balls / 6
            run_rate = total_runs / overs if overs > 0 else 0

            team_performance.append({
                "match_id": match_id,
                "team": team,
                "runs": total_runs,
                "wickets": wickets,
                "legal_balls": legal_balls,
                "run_rate": run_rate
            })

    except Exception:
        continue

odi_performance = pd.DataFrame(team_performance)

print("Performance records:", len(odi_performance))
odi_performance.head()

Performance records: 5086


,match_id,team,runs,wickets,legal_balls,run_rate
0,1050227,Pakistan,308,6,300,6.160000
1,1050227,West Indies,172,10,264,3.909091
2,64849,Zimbabwe,229,5,300,4.580000
3,64849,West Indies,208,10,284,4.394366
4,1348657,India,117,10,156,4.500000


In [ ]:
team1_perf = odi_performance.rename(columns={
    "team": "team_1",
    "runs": "team_1_runs",
    "wickets": "team_1_wickets",
    "legal_balls": "team_1_legal_balls",
    "run_rate": "team_1_run_rate"
})

team2_perf = odi_performance.rename(columns={
    "team": "team_2",
    "runs": "team_2_runs",
    "wickets": "team_2_wickets",
    "legal_balls": "team_2_legal_balls",
    "run_rate": "team_2_run_rate"
})

odi_matches = odi_matches.merge(
    team1_perf[
        ["match_id", "team_1",
         "team_1_runs", "team_1_wickets",
         "team_1_legal_balls", "team_1_run_rate"]
    ],
    on=["match_id", "team_1"],
    how="left"
)

odi_matches = odi_matches.merge(
    team2_perf[
        ["match_id", "team_2",
         "team_2_runs", "team_2_wickets",
         "team_2_legal_balls", "team_2_run_rate"]
    ],
    on=["match_id", "team_2"],
    how="left"
)

print("Dataset shape:", odi_matches.shape)
odi_matches.head()

Dataset shape: (2442, 21)


,match_id,date,season,venue,city,team_1,team_2,winner,match_type,gender,...,toss_winner,toss_decision,team_1_runs,team_1_wickets,team_1_legal_balls,team_1_run_rate,team_2_runs,team_2_wickets,team_2_legal_balls,team_2_run_rate
0,66283,2002-06-27,2002,"Trent Bridge, Nottingham",Nottingham,England,Sri Lanka,England,ODI,male,...,England,bat,293,6,300,5.860000,249,9,300,4.980000
1,66284,2002-06-29,2002,"Lord's, London",London,England,India,India,ODI,male,...,England,bat,271,7,300,5.420000,272,4,293,5.569966
2,64814,2002-12-29,2002/03,"McLean Park, Napier",Napier,New Zealand,India,New Zealand,ODI,male,...,India,field,254,9,300,5.080000,219,10,262,5.015267
3,64815,2003-01-01,2002/03,"Jade Stadium, Christchurch",Christchurch,India,New Zealand,New Zealand,ODI,male,...,India,bat,108,10,247,2.623482,109,5,161,4.062112
4,64816,2003-01-04,2002/03,"Davies Park, Queenstown",Queenstown,India,New Zealand,New Zealand,ODI,male,...,New Zealand,field,122,10,262,2.793893,123,3,154,4.792208


In [ ]:
odi_matches["target"] = (
    odi_matches["winner"] == odi_matches["team_1"]
).astype(int)

print("Target distribution:")
print(odi_matches["target"].value_counts())

print("\nColumns:")
print(odi_matches.columns.tolist())

Target distribution:
target
1    1331
0    1111
Name: count, dtype: int64

Columns:
['match_id', 'date', 'season', 'venue', 'city', 'team_1', 'team_2', 'winner', 'match_type', 'gender', 'event', 'toss_winner', 'toss_decision', 'team_1_runs', 'team_1_wickets', 'team_1_legal_balls', 'team_1_run_rate', 'team_2_runs', 'team_2_wickets', 'team_2_legal_balls', 'team_2_run_rate', 'target']


In [ ]:
from collections import defaultdict

team_stats = defaultdict(lambda: {
    "matches": 0,
    "wins": 0,
    "runs": [],
    "wickets": [],
    "run_rates": []
})

historical_rows = []

for _, row in odi_matches.iterrows():

    team1 = row["team_1"]
    team2 = row["team_2"]

    s1 = team_stats[team1]
    s2 = team_stats[team2]

    historical_rows.append({
        "team_1_win_rate": s1["wins"] / s1["matches"] if s1["matches"] else 0.5,
        "team_2_win_rate": s2["wins"] / s2["matches"] if s2["matches"] else 0.5,

        "team_1_avg_runs": sum(s1["runs"]) / len(s1["runs"]) if s1["runs"] else 0,
        "team_2_avg_runs": sum(s2["runs"]) / len(s2["runs"]) if s2["runs"] else 0,

        "team_1_avg_wickets": sum(s1["wickets"]) / len(s1["wickets"]) if s1["wickets"] else 0,
        "team_2_avg_wickets": sum(s2["wickets"]) / len(s2["wickets"]) if s2["wickets"] else 0,

        "team_1_avg_run_rate": sum(s1["run_rates"]) / len(s1["run_rates"]) if s1["run_rates"] else 0,
        "team_2_avg_run_rate": sum(s2["run_rates"]) / len(s2["run_rates"]) if s2["run_rates"] else 0
    })

    # Update AFTER recording historical stats
    if pd.notna(row["team_1_runs"]):
        s1["runs"].append(row["team_1_runs"])
        s1["wickets"].append(row["team_1_wickets"])
        s1["run_rates"].append(row["team_1_run_rate"])

    if pd.notna(row["team_2_runs"]):
        s2["runs"].append(row["team_2_runs"])
        s2["wickets"].append(row["team_2_wickets"])
        s2["run_rates"].append(row["team_2_run_rate"])

    s1["matches"] += 1
    s2["matches"] += 1

    if row["winner"] == team1:
        s1["wins"] += 1
    elif row["winner"] == team2:
        s2["wins"] += 1

historical_df = pd.DataFrame(historical_rows)

odi_matches = pd.concat(
    [odi_matches.reset_index(drop=True), historical_df],
    axis=1
)

print("Historical features added.")
odi_matches.head()

Historical features added.


,match_id,date,season,venue,city,team_1,team_2,winner,match_type,gender,...,team_2_run_rate,target,team_1_win_rate,team_2_win_rate,team_1_avg_runs,team_2_avg_runs,team_1_avg_wickets,team_2_avg_wickets,team_1_avg_run_rate,team_2_avg_run_rate
0,66283,2002-06-27,2002,"Trent Bridge, Nottingham",Nottingham,England,Sri Lanka,England,ODI,male,...,4.980000,1,0.500000,0.5,0.000000,0.0,0.0,0.0,0.000000,0.000000
1,66284,2002-06-29,2002,"Lord's, London",London,England,India,India,ODI,male,...,5.569966,0,1.000000,0.5,293.000000,0.0,6.0,0.0,5.860000,0.000000
2,64814,2002-12-29,2002/03,"McLean Park, Napier",Napier,New Zealand,India,New Zealand,ODI,male,...,5.015267,1,0.500000,1.0,0.000000,272.0,0.0,4.0,0.000000,5.569966
3,64815,2003-01-01,2002/03,"Jade Stadium, Christchurch",Christchurch,India,New Zealand,New Zealand,ODI,male,...,4.062112,0,0.500000,1.0,245.500000,254.0,7.0,9.0,5.292617,5.080000
4,64816,2003-01-04,2002/03,"Davies Park, Queenstown",Queenstown,India,New Zealand,New Zealand,ODI,male,...,4.792208,0,0.333333,1.0,199.666667,181.5,8.0,7.0,4.402905,4.571056


In [ ]:
odi_matches["win_rate_diff"] = (
    odi_matches["team_1_win_rate"] -
    odi_matches["team_2_win_rate"]
)

odi_matches["avg_runs_diff"] = (
    odi_matches["team_1_avg_runs"] -
    odi_matches["team_2_avg_runs"]
)

odi_matches["avg_wickets_diff"] = (
    odi_matches["team_1_avg_wickets"] -
    odi_matches["team_2_avg_wickets"]
)

odi_matches["run_rate_diff"] = (
    odi_matches["team_1_avg_run_rate"] -
    odi_matches["team_2_avg_run_rate"]
)

print(odi_matches[
    [
        "team_1", "team_2",
        "win_rate_diff",
        "avg_runs_diff",
        "avg_wickets_diff",
        "run_rate_diff"
    ]
].head())

        team_1       team_2  win_rate_diff  avg_runs_diff  avg_wickets_diff  \
0      England    Sri Lanka       0.000000       0.000000               0.0   
1      England        India       0.500000     293.000000               6.0   
2  New Zealand        India      -0.500000    -272.000000              -4.0   
3        India  New Zealand      -0.500000      -8.500000              -2.0   
4        India  New Zealand      -0.666667      18.166667               1.0   

   run_rate_diff  
0       0.000000  
1       5.860000  
2      -5.569966  
3       0.212617  
4      -0.168151  


In [ ]:
recent_results = defaultdict(list)

recent_form_rows = []

for _, row in odi_matches.iterrows():

    team1 = row["team_1"]
    team2 = row["team_2"]

    team1_form = recent_results[team1][-5:]
    team2_form = recent_results[team2][-5:]

    team1_recent = (
        sum(team1_form) / len(team1_form)
        if team1_form else 0.5
    )

    team2_recent = (
        sum(team2_form) / len(team2_form)
        if team2_form else 0.5
    )

    recent_form_rows.append({
        "team_1_recent_form": team1_recent,
        "team_2_recent_form": team2_recent
    })

    # Update AFTER calculating the feature
    result1 = 1 if row["winner"] == team1 else 0
    result2 = 1 if row["winner"] == team2 else 0

    recent_results[team1].append(result1)
    recent_results[team2].append(result2)

recent_df = pd.DataFrame(recent_form_rows)

odi_matches = pd.concat(
    [odi_matches.reset_index(drop=True), recent_df],
    axis=1
)

odi_matches["recent_form_diff"] = (
    odi_matches["team_1_recent_form"] -
    odi_matches["team_2_recent_form"]
)

print("Recent form added.")

Recent form added.


In [ ]:
h2h = defaultdict(lambda: {
    "matches": 0,
    "team1_wins": 0
})

h2h_rows = []

for _, row in odi_matches.iterrows():

    team1 = row["team_1"]
    team2 = row["team_2"]

    key = tuple(sorted([team1, team2]))

    record = h2h[key]

    if record["matches"] == 0:
        h2h_rate = 0.5
    else:
        # Calculate from team_1's perspective
        if team1 == key[0]:
            h2h_rate = record["team1_wins"] / record["matches"]
        else:
            h2h_rate = (
                1 - record["team1_wins"] / record["matches"]
            )

    h2h_rows.append({
        "team_1_h2h_win_rate": h2h_rate
    })

    record["matches"] += 1

    if row["winner"] == key[0]:
        record["team1_wins"] += 1

h2h_df = pd.DataFrame(h2h_rows)

odi_matches = pd.concat(
    [odi_matches.reset_index(drop=True), h2h_df],
    axis=1
)

print("H2H feature added.")

H2H feature added.


In [ ]:
elo = defaultdict(lambda: 1500)

elo_rows = []

K = 20

for _, row in odi_matches.iterrows():

    team1 = row["team_1"]
    team2 = row["team_2"]

    elo1 = elo[team1]
    elo2 = elo[team2]

    elo_rows.append({
        "team_1_elo": elo1,
        "team_2_elo": elo2,
        "elo_diff": elo1 - elo2
    })

    expected1 = 1 / (
        1 + 10 ** ((elo2 - elo1) / 400)
    )

    actual1 = 1 if row["winner"] == team1 else 0

    elo[team1] = elo1 + K * (actual1 - expected1)
    elo[team2] = elo2 + K * ((1 - actual1) - (1 - expected1))

elo_df = pd.DataFrame(elo_rows)

odi_matches = pd.concat(
    [odi_matches.reset_index(drop=True), elo_df],
    axis=1
)

print("Elo ratings added.")
print("Teams tracked:", len(elo))

Elo ratings added.
Teams tracked: 27


In [ ]:
odi_features = [
    "elo_diff",
    "win_rate_diff",
    "recent_form_diff",
    "avg_runs_diff",
    "avg_wickets_diff",
    "run_rate_diff",
    "team_1_h2h_win_rate"
]

print("ODI features:")
for i, feature in enumerate(odi_features, 1):
    print(f"{i}. {feature}")

print("\nMissing values:")
print(odi_matches[odi_features].isna().sum())

print("\nDataset shape:", odi_matches.shape)

ODI features:
1. elo_diff
2. win_rate_diff
3. recent_form_diff
4. avg_runs_diff
5. avg_wickets_diff
6. run_rate_diff
7. team_1_h2h_win_rate

Missing values:
elo_diff               0
win_rate_diff          0
recent_form_diff       0
avg_runs_diff          0
avg_wickets_diff       0
run_rate_diff          0
team_1_h2h_win_rate    0
dtype: int64

Dataset shape: (2442, 41)


In [ ]:
from sklearn.model_selection import train_test_split

# Remove rows with missing feature values
odi_ml = odi_matches.dropna(
    subset=odi_features + ["target"]
).copy()

X = odi_ml[odi_features]
y = odi_ml["target"]

print("Total usable matches:", len(odi_ml))
print("Features:", X.shape)
print("Target distribution:")
print(y.value_counts())

Total usable matches: 2442
Features: (2442, 7)
Target distribution:
target
1    1331
0    1111
Name: count, dtype: int64


In [ ]:
split1 = int(len(odi_ml) * 0.70)
split2 = int(len(odi_ml) * 0.85)

X_train = X.iloc[:split1]
y_train = y.iloc[:split1]

X_val = X.iloc[split1:split2]
y_val = y.iloc[split1:split2]

X_test = X.iloc[split2:]
y_test = y.iloc[split2:]

print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))

print("\nDate ranges:")
print("Train:", odi_ml.iloc[0]["date"], "→", odi_ml.iloc[split1-1]["date"])
print("Val:", odi_ml.iloc[split1]["date"], "→", odi_ml.iloc[split2-1]["date"])
print("Test:", odi_ml.iloc[split2]["date"], "→", odi_ml.iloc[-1]["date"])

Training: 1709
Validation: 366
Testing: 367

Date ranges:
Train: 2002-06-27 00:00:00 → 2019-06-14 00:00:00
Val: 2019-06-15 00:00:00 → 2023-05-05 00:00:00
Test: 2023-05-07 00:00:00 → 2026-09-09 00:00:00


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

lr_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

lr_model.fit(X_train, y_train)

print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("Random Forest trained successfully.")

Random Forest trained successfully.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score
)

lr_prob = lr_model.predict_proba(X_test)[:, 1]
rf_prob = rf_model.predict_proba(X_test)[:, 1]

lr_pred = (lr_prob >= 0.50).astype(int)
rf_pred = (rf_prob >= 0.50).astype(int)

print("===== LOGISTIC REGRESSION =====")
print("Accuracy:", round(accuracy_score(y_test, lr_pred), 4))
print("F1 Score:", round(f1_score(y_test, lr_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, lr_prob), 4))

print("\n===== RANDOM FOREST =====")
print("Accuracy:", round(accuracy_score(y_test, rf_pred), 4))
print("F1 Score:", round(f1_score(y_test, rf_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, rf_prob), 4))

===== LOGISTIC REGRESSION =====
Accuracy: 0.6158
F1 Score: 0.6569
ROC-AUC: 0.6553

===== RANDOM FOREST =====
Accuracy: 0.594
F1 Score: 0.659
ROC-AUC: 0.6608


In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_score(y_test, lr_pred),
        accuracy_score(y_test, rf_pred)
    ],
    "F1 Score": [
        f1_score(y_test, lr_pred),
        f1_score(y_test, rf_pred)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, lr_prob),
        roc_auc_score(y_test, rf_prob)
    ]
})

comparison = comparison.round(4)

print(comparison)

                 Model  Accuracy  F1 Score  ROC-AUC
0  Logistic Regression    0.6158    0.6569   0.6553
1        Random Forest    0.5940    0.6590   0.6608


In [ ]:
import numpy as np

# Validation probabilities
lr_val_prob = lr_model.predict_proba(X_val)[:, 1]
rf_val_prob = rf_model.predict_proba(X_val)[:, 1]

print("Validation probabilities generated.")
print("LR sample:", lr_val_prob[:5])
print("RF sample:", rf_val_prob[:5])

Validation probabilities generated.
LR sample: [0.81511812 0.79201897 0.61016397 0.50870413 0.58501909]
RF sample: [0.86880695 0.79158874 0.65431799 0.45586431 0.51961882]


In [ ]:
thresholds = np.arange(0.40, 0.71, 0.01)

threshold_results = []

for threshold in thresholds:

    val_pred = (lr_val_prob >= threshold).astype(int)

    accuracy = accuracy_score(y_val, val_pred)

    threshold_results.append({
        "threshold": threshold,
        "accuracy": accuracy
    })

threshold_df = pd.DataFrame(threshold_results)

best_row = threshold_df.loc[
    threshold_df["accuracy"].idxmax()
]

best_threshold = float(best_row["threshold"])

print("Best threshold:", best_threshold)
print("Validation accuracy:", round(best_row["accuracy"], 4))

Best threshold: 0.4900000000000001
Validation accuracy: 0.6585


In [ ]:
lr_test_pred_tuned = (
    lr_prob >= best_threshold
).astype(int)

final_accuracy = accuracy_score(
    y_test,
    lr_test_pred_tuned
)

final_f1 = f1_score(
    y_test,
    lr_test_pred_tuned
)

final_auc = roc_auc_score(
    y_test,
    lr_prob
)

print("===== FINAL ODI LOGISTIC REGRESSION =====")
print("Threshold:", best_threshold)
print("Accuracy:", round(final_accuracy, 4))
print("F1 Score:", round(final_f1, 4))
print("ROC-AUC:", round(final_auc, 4))

===== FINAL ODI LOGISTIC REGRESSION =====
Threshold: 0.4900000000000001
Accuracy: 0.6104
F1 Score: 0.6603
ROC-AUC: 0.6553


In [ ]:
odi_model_summary = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "F1 Score",
        "ROC-AUC",
        "Threshold"
    ],
    "Value": [
        final_accuracy,
        final_f1,
        final_auc,
        best_threshold
    ]
})

odi_model_summary["Value"] = odi_model_summary["Value"].round(4)

print(odi_model_summary)

      Metric   Value
0   Accuracy  0.6104
1   F1 Score  0.6603
2    ROC-AUC  0.6553
3  Threshold  0.4900


In [ ]:
lr_coefficients = lr_model.named_steps["model"].coef_[0]

feature_coefficients = pd.DataFrame({
    "Feature": odi_features,
    "Coefficient": lr_coefficients
})

feature_coefficients["Absolute_Impact"] = (
    feature_coefficients["Coefficient"].abs()
)

feature_coefficients = feature_coefficients.sort_values(
    "Absolute_Impact",
    ascending=False
)

print(feature_coefficients)

               Feature  Coefficient  Absolute_Impact
0             elo_diff     0.468332         0.468332
3        avg_runs_diff     0.354120         0.354120
4     avg_wickets_diff    -0.183400         0.183400
5        run_rate_diff     0.153080         0.153080
1        win_rate_diff    -0.112169         0.112169
2     recent_form_diff     0.102759         0.102759
6  team_1_h2h_win_rate     0.082894         0.082894


In [ ]:
import joblib
import json

joblib.dump(
    lr_model,
    "odi_match_outcome_model.pkl"
)

joblib.dump(
    best_threshold,
    "odi_prediction_threshold.pkl"
)

joblib.dump(
    odi_features,
    "odi_model_features.pkl"
)

odi_matches.to_csv(
    "odi_match_data_for_prediction.csv",
    index=False
)

feature_coefficients.to_csv(
    "odi_feature_coefficients.csv",
    index=False
)

model_info = {
    "model_name": "MATCHAI ODI Match Outcome Predictor",
    "algorithm": "Logistic Regression",
    "dataset": "Men's ODI Internationals",
    "features": odi_features,
    "threshold": best_threshold,
    "accuracy": float(final_accuracy),
    "f1_score": float(final_f1),
    "roc_auc": float(final_auc),
    "initial_elo": 1500,
    "elo_k_factor": 20,
    "recent_form_matches": 5
}

with open("odi_model_info.json", "w") as f:
    json.dump(model_info, f, indent=4)

print("All ODI model assets saved successfully.")

All ODI model assets saved successfully.


In [ ]:
import zipfile
import os

odi_files = [
    "odi_match_outcome_model.pkl",
    "odi_prediction_threshold.pkl",
    "odi_model_features.pkl",
    "odi_match_data_for_prediction.csv",
    "odi_feature_coefficients.csv",
    "odi_model_info.json"
]

with zipfile.ZipFile(
    "MATCHAI_ODI_Model.zip",
    "w",
    zipfile.ZIP_DEFLATED
) as zipf:

    for file in odi_files:
        if os.path.exists(file):
            zipf.write(file)

print("MATCHAI_ODI_Model.zip created successfully.")

for file in odi_files:
    print("✓", file)

MATCHAI_ODI_Model.zip created successfully.
✓ odi_match_outcome_model.pkl
✓ odi_prediction_threshold.pkl
✓ odi_model_features.pkl
✓ odi_match_data_for_prediction.csv
✓ odi_feature_coefficients.csv
✓ odi_model_info.json


In [ ]:
from google.colab import files

files.download("MATCHAI_ODI_Model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from collections import defaultdict

odi_team_stats = defaultdict(lambda: {
    "matches": 0,
    "wins": 0,
    "runs": [],
    "wickets": [],
    "run_rates": [],
    "results": []
})

for _, row in odi_matches.sort_values("date").iterrows():

    team1 = row["team_1"]
    team2 = row["team_2"]

    # Team 1
    odi_team_stats[team1]["matches"] += 1
    odi_team_stats[team1]["runs"].append(row["team_1_runs"])
    odi_team_stats[team1]["wickets"].append(row["team_1_wickets"])
    odi_team_stats[team1]["run_rates"].append(row["team_1_run_rate"])

    result1 = 1 if row["winner"] == team1 else 0
    odi_team_stats[team1]["wins"] += result1
    odi_team_stats[team1]["results"].append(result1)

    # Team 2
    odi_team_stats[team2]["matches"] += 1
    odi_team_stats[team2]["runs"].append(row["team_2_runs"])
    odi_team_stats[team2]["wickets"].append(row["team_2_wickets"])
    odi_team_stats[team2]["run_rates"].append(row["team_2_run_rate"])

    result2 = 1 if row["winner"] == team2 else 0
    odi_team_stats[team2]["wins"] += result2
    odi_team_stats[team2]["results"].append(result2)

print("Teams available:", len(odi_team_stats))

odi_teams = sorted(odi_team_stats.keys())

print("\nODI Teams:")
for team in odi_teams:
    print("-", team)

Teams available: 27

ODI Teams:
- Africa XI
- Asia XI
- Australia
- Bangladesh
- Bermuda
- Canada
- England
- Hong Kong
- ICC World XI
- India
- Ireland
- Jersey
- Kenya
- Namibia
- Nepal
- Netherlands
- New Zealand
- Oman
- Pakistan
- Papua New Guinea
- Scotland
- South Africa
- Sri Lanka
- United Arab Emirates
- United States of America
- West Indies
- Zimbabwe


In [ ]:
odi_elo = defaultdict(lambda: 1500)

K = 20

for _, row in odi_matches.sort_values("date").iterrows():

    team1 = row["team_1"]
    team2 = row["team_2"]

    elo1 = odi_elo[team1]
    elo2 = odi_elo[team2]

    expected1 = 1 / (
        1 + 10 ** ((elo2 - elo1) / 400)
    )

    actual1 = 1 if row["winner"] == team1 else 0

    odi_elo[team1] = elo1 + K * (
        actual1 - expected1
    )

    odi_elo[team2] = elo2 + K * (
        (1 - actual1) - (1 - expected1)
    )

print("Final ODI Elo ratings calculated.")
print("Teams:", len(odi_elo))

Final ODI Elo ratings calculated.
Teams: 27


In [ ]:
odi_h2h = defaultdict(lambda: {
    "matches": 0,
    "wins": defaultdict(int)
})

for _, row in odi_matches.sort_values("date").iterrows():

    team1 = row["team_1"]
    team2 = row["team_2"]

    key = tuple(sorted([team1, team2]))

    odi_h2h[key]["matches"] += 1
    odi_h2h[key]["wins"][row["winner"]] += 1

print("H2H database created.")
print("Team pair records:", len(odi_h2h))

H2H database created.
Team pair records: 183


In [ ]:
def predict_odi_match(team1, team2):

    if team1 not in odi_team_stats:
        return f"Team not found: {team1}"

    if team2 not in odi_team_stats:
        return f"Team not found: {team2}"

    if team1 == team2:
        return "Please select two different teams."

    s1 = odi_team_stats[team1]
    s2 = odi_team_stats[team2]

    # Win rate
    win_rate_1 = (
        s1["wins"] / s1["matches"]
        if s1["matches"] > 0 else 0.5
    )

    win_rate_2 = (
        s2["wins"] / s2["matches"]
        if s2["matches"] > 0 else 0.5
    )

    # Recent form
    recent1 = s1["results"][-5:]
    recent2 = s2["results"][-5:]

    recent_form_1 = (
        sum(recent1) / len(recent1)
        if recent1 else 0.5
    )

    recent_form_2 = (
        sum(recent2) / len(recent2)
        if recent2 else 0.5
    )

    # Average performance
    avg_runs_1 = sum(s1["runs"]) / len(s1["runs"])
    avg_runs_2 = sum(s2["runs"]) / len(s2["runs"])

    avg_wickets_1 = sum(s1["wickets"]) / len(s1["wickets"])
    avg_wickets_2 = sum(s2["wickets"]) / len(s2["wickets"])

    avg_rr_1 = sum(s1["run_rates"]) / len(s1["run_rates"])
    avg_rr_2 = sum(s2["run_rates"]) / len(s2["run_rates"])

    # Elo
    elo1 = odi_elo[team1]
    elo2 = odi_elo[team2]

    # H2H
    key = tuple(sorted([team1, team2]))
    h2h_record = odi_h2h.get(key)

    if h2h_record and h2h_record["matches"] > 0:
        h2h_1 = (
            h2h_record["wins"][team1] /
            h2h_record["matches"]
        )
    else:
        h2h_1 = 0.5

    # Final feature vector
    features = pd.DataFrame([{
        "elo_diff": elo1 - elo2,
        "win_rate_diff": win_rate_1 - win_rate_2,
        "recent_form_diff": recent_form_1 - recent_form_2,
        "avg_runs_diff": avg_runs_1 - avg_runs_2,
        "avg_wickets_diff": avg_wickets_1 - avg_wickets_2,
        "run_rate_diff": avg_rr_1 - avg_rr_2,
        "team_1_h2h_win_rate": h2h_1
    }])

    # Ensure exact feature order
    features = features[odi_features]

    probability_team1 = lr_model.predict_proba(features)[0][1]

    prediction = (
        team1
        if probability_team1 >= best_threshold
        else team2
    )

    return {
        "team_1": team1,
        "team_2": team2,
        "predicted_winner": prediction,
        "team_1_probability": round(
            probability_team1 * 100, 2
        ),
        "team_2_probability": round(
            (1 - probability_team1) * 100, 2
        ),
        "threshold": best_threshold,
        "features": features.iloc[0].to_dict()
    }

In [ ]:
print("Available teams:")
print(odi_teams)

Available teams:
['Africa XI', 'Asia XI', 'Australia', 'Bangladesh', 'Bermuda', 'Canada', 'England', 'Hong Kong', 'ICC World XI', 'India', 'Ireland', 'Jersey', 'Kenya', 'Namibia', 'Nepal', 'Netherlands', 'New Zealand', 'Oman', 'Pakistan', 'Papua New Guinea', 'Scotland', 'South Africa', 'Sri Lanka', 'United Arab Emirates', 'United States of America', 'West Indies', 'Zimbabwe']


In [ ]:
# Change these to any two teams from the list above

team1 = odi_teams[0]
team2 = odi_teams[1]

result = predict_odi_match(team1, team2)

print("===== MATCHAI ODI PREDICTION =====")
print("Team 1:", result["team_1"])
print("Team 2:", result["team_2"])
print("Predicted Winner:", result["predicted_winner"])
print(
    result["team_1"],
    "Probability:",
    result["team_1_probability"],
    "%"
)
print(
    result["team_2"],
    "Probability:",
    result["team_2_probability"],
    "%"
)
print("Threshold:", result["threshold"])

===== MATCHAI ODI PREDICTION =====
Team 1: Africa XI
Team 2: Asia XI
Predicted Winner: Asia XI
Africa XI Probability: 46.08 %
Asia XI Probability: 53.92 %
Threshold: 0.4900000000000001


In [ ]:
print("===== MODEL DECISION FEATURES =====")

for feature, value in result["features"].items():
    print(f"{feature}: {value:.4f}")

===== MODEL DECISION FEATURES =====
elo_diff: -39.8716
win_rate_diff: -0.5000
recent_form_diff: -0.5000
avg_runs_diff: -19.0000
avg_wickets_diff: 0.7500
run_rate_diff: -0.2310
team_1_h2h_win_rate: 0.2500


In [ ]:
print("===== MATCHAI ODI MODEL =====")
print("Algorithm: Logistic Regression")
print("Dataset: Men's ODI Internationals")
print("Accuracy:", round(final_accuracy * 100, 2), "%")
print("F1 Score:", round(final_f1 * 100, 2), "%")
print("ROC-AUC:", round(final_auc, 4))
print("Prediction Threshold:", best_threshold)
print("Features:", len(odi_features))

===== MATCHAI ODI MODEL =====
Algorithm: Logistic Regression
Dataset: Men's ODI Internationals
Accuracy: 61.04 %
F1 Score: 66.03 %
ROC-AUC: 0.6553
Prediction Threshold: 0.4900000000000001
Features: 7


In [ ]:
import os

print("===== ODI MODEL FILES =====")

for file in [
    "odi_match_outcome_model.pkl",
    "odi_prediction_threshold.pkl",
    "odi_model_features.pkl",
    "odi_match_data_for_prediction.csv",
    "odi_feature_coefficients.csv",
    "odi_model_info.json",
    "MATCHAI_ODI_Model.zip"
]:
    if os.path.exists(file):
        size = os.path.getsize(file) / (1024 * 1024)
        print(f"✓ {file} ({size:.2f} MB)")
    else:
        print(f"✗ Missing: {file}")

===== ODI MODEL FILES =====
✓ odi_match_outcome_model.pkl (0.00 MB)
✓ odi_prediction_threshold.pkl (0.00 MB)
✓ odi_model_features.pkl (0.00 MB)
✓ odi_match_data_for_prediction.csv (1.15 MB)
✓ odi_feature_coefficients.csv (0.00 MB)
✓ odi_model_info.json (0.00 MB)
✓ MATCHAI_ODI_Model.zip (0.41 MB)


In [ ]:
print("""
========================================
       MATCHAI ODI MODEL READY
========================================

Model        : Logistic Regression
Dataset      : Men's ODI Internationals
Features     : 7
Recent Form  : Last 5 Matches
Elo          : Enabled
H2H          : Enabled
Threshold    : Tuned on Validation Set

Model Accuracy : {:.2f}%
F1 Score       : {:.2f}%
ROC-AUC        : {:.4f}

Prediction Engine : READY
========================================
""".format(
    final_accuracy * 100,
    final_f1 * 100,
    final_auc
))


       MATCHAI ODI MODEL READY

Model        : Logistic Regression
Dataset      : Men's ODI Internationals
Features     : 7
Recent Form  : Last 5 Matches
Elo          : Enabled
H2H          : Enabled
Threshold    : Tuned on Validation Set

Model Accuracy : 61.04%
F1 Score       : 66.03%
ROC-AUC        : 0.6553

Prediction Engine : READY



In [ ]:
# ==========================================
# ODI FEATURE EXPERIMENT — CELL 1
# CHECK AVAILABLE COLUMNS
# ==========================================

print("ODI dataset shape:", odi_matches.shape)

print("\nODI columns:")
for col in odi_matches.columns:
    print("✓", col)

print("\nRequired columns check:")

required = [
    "team_1",
    "team_2",
    "venue",
    "toss_winner",
    "toss_decision",
    "target"
]

for col in required:
    print(
        f"{'✓' if col in odi_matches.columns else '✗'} {col}"
    )

ODI dataset shape: (2442, 41)

ODI columns:
✓ match_id
✓ date
✓ season
✓ venue
✓ city
✓ team_1
✓ team_2
✓ winner
✓ match_type
✓ gender
✓ event
✓ toss_winner
✓ toss_decision
✓ team_1_runs
✓ team_1_wickets
✓ team_1_legal_balls
✓ team_1_run_rate
✓ team_2_runs
✓ team_2_wickets
✓ team_2_legal_balls
✓ team_2_run_rate
✓ target
✓ team_1_win_rate
✓ team_2_win_rate
✓ team_1_avg_runs
✓ team_2_avg_runs
✓ team_1_avg_wickets
✓ team_2_avg_wickets
✓ team_1_avg_run_rate
✓ team_2_avg_run_rate
✓ win_rate_diff
✓ avg_runs_diff
✓ avg_wickets_diff
✓ run_rate_diff
✓ team_1_recent_form
✓ team_2_recent_form
✓ recent_form_diff
✓ team_1_h2h_win_rate
✓ team_1_elo
✓ team_2_elo
✓ elo_diff

Required columns check:
✓ team_1
✓ team_2
✓ venue
✓ toss_winner
✓ toss_decision
✓ target


In [ ]:
# ==========================================
# ODI FEATURE EXPERIMENT — CELL 2
# VENUE PERFORMANCE
# ==========================================

from collections import defaultdict

odi_venue_history = defaultdict(
    lambda: {
        "matches": 0,
        "wins": 0
    }
)

odi_venue_rates = []

odi_sorted = odi_matches.sort_values("date").copy()

for _, row in odi_sorted.iterrows():

    team1 = row["team_1"]
    team2 = row["team_2"]
    venue = row["venue"]

    # Historical performance BEFORE this match
    stats1 = odi_venue_history[(team1, venue)]
    stats2 = odi_venue_history[(team2, venue)]

    rate1 = (
        stats1["wins"] / stats1["matches"]
        if stats1["matches"] > 0
        else 0.5
    )

    rate2 = (
        stats2["wins"] / stats2["matches"]
        if stats2["matches"] > 0
        else 0.5
    )

    odi_venue_rates.append(
        rate1 - rate2
    )

    # Update history AFTER calculating feature
    stats1["matches"] += 1
    stats2["matches"] += 1

    if row["target"] == 1:
        stats1["wins"] += 1
    else:
        stats2["wins"] += 1

odi_sorted["venue_win_rate_diff"] = odi_venue_rates

odi_matches = odi_sorted.reset_index(drop=True)

print("Venue feature created successfully.")
print("Feature: venue_win_rate_diff")

print("\nSample:")
display(
    odi_matches[
        ["team_1", "team_2", "venue",
         "venue_win_rate_diff", "target"]
    ].head()
)

Venue feature created successfully.
Feature: venue_win_rate_diff

Sample:


,team_1,team_2,venue,venue_win_rate_diff,target
0,England,Sri Lanka,"Trent Bridge, Nottingham",0.0,1
1,England,India,"Lord's, London",0.0,0
2,New Zealand,India,"McLean Park, Napier",0.0,1
3,India,New Zealand,"Jade Stadium, Christchurch",0.0,0
4,India,New Zealand,"Davies Park, Queenstown",0.0,0


In [ ]:
# ==========================================
# ODI FEATURE EXPERIMENT — CELL 3
# TOSS FEATURES
# ==========================================

def calculate_toss_features(row):

    team1 = row["team_1"]
    team2 = row["team_2"]

    toss_winner = row["toss_winner"]
    decision = str(row["toss_decision"]).lower()

    # Toss winner from Team 1 perspective
    if toss_winner == team1:
        toss_diff = 1
    elif toss_winner == team2:
        toss_diff = -1
    else:
        toss_diff = 0

    # Team that bats first from Team 1 perspective
    if toss_winner == team1:

        if decision == "bat":
            bat_first_diff = 1
        else:
            bat_first_diff = -1

    elif toss_winner == team2:

        if decision == "bat":
            bat_first_diff = -1
        else:
            bat_first_diff = 1

    else:
        bat_first_diff = 0

    return toss_diff, bat_first_diff


odi_toss_values = odi_matches.apply(
    calculate_toss_features,
    axis=1
)

odi_matches["toss_winner_diff"] = [
    x[0] for x in odi_toss_values
]

odi_matches["bat_first_diff"] = [
    x[1] for x in odi_toss_values
]

print("Toss features created successfully.")

print(
    odi_matches[
        [
            "toss_winner_diff",
            "bat_first_diff"
        ]
    ].head()
)

Toss features created successfully.
   toss_winner_diff  bat_first_diff
0                 1               1
1                 1               1
2                -1               1
3                 1               1
4                -1               1


In [ ]:
# ==========================================
# ODI FEATURE EXPERIMENT — CELL 4
# BASELINE + VENUE
# ==========================================

odi_features_venue = odi_features + [
    "venue_win_rate_diff"
]

X_odi_venue = odi_matches[
    odi_features_venue
].copy()

y_odi_venue = odi_matches["target"].copy()

n = len(odi_matches)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_odi_venue_train = X_odi_venue.iloc[:train_end]
X_odi_venue_val = X_odi_venue.iloc[train_end:val_end]
X_odi_venue_test = X_odi_venue.iloc[val_end:]

y_odi_venue_train = y_odi_venue.iloc[:train_end]
y_odi_venue_val = y_odi_venue.iloc[train_end:val_end]
y_odi_venue_test = y_odi_venue.iloc[val_end:]

print("ODI + VENUE DATASET")
print("=" * 50)

print("Features:", len(odi_features_venue))
print("Train:", len(X_odi_venue_train))
print("Validation:", len(X_odi_venue_val))
print("Test:", len(X_odi_venue_test))

ODI + VENUE DATASET
Features: 8
Train: 1709
Validation: 366
Test: 367


In [ ]:
# ==========================================
# ODI FEATURE EXPERIMENT — CELL 5
# LOGISTIC REGRESSION + VENUE
# ==========================================

lr_odi_venue = Pipeline([
    ("scaler", StandardScaler()),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            random_state=42
        )
    )
])

lr_odi_venue.fit(
    X_odi_venue_train,
    y_odi_venue_train
)

lr_odi_venue_pred = lr_odi_venue.predict(
    X_odi_venue_test
)

lr_odi_venue_proba = lr_odi_venue.predict_proba(
    X_odi_venue_test
)

print("ODI Logistic Regression + Venue trained successfully.")

ODI Logistic Regression + Venue trained successfully.


In [ ]:
# ==========================================
# ODI FEATURE EXPERIMENT — CELL 6
# EVALUATE VENUE
# ==========================================

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report
)

accuracy = accuracy_score(
    y_odi_venue_test,
    lr_odi_venue_pred
)

macro_f1 = f1_score(
    y_odi_venue_test,
    lr_odi_venue_pred,
    average="macro"
)

weighted_f1 = f1_score(
    y_odi_venue_test,
    lr_odi_venue_pred,
    average="weighted"
)

roc_auc = roc_auc_score(
    y_odi_venue_test,
    lr_odi_venue_proba[:, 1]
)

print("=" * 60)
print("ODI — LOGISTIC REGRESSION + VENUE")
print("=" * 60)

print(f"Accuracy    : {accuracy:.4f}")
print(f"Macro F1    : {macro_f1:.4f}")
print(f"Weighted F1 : {weighted_f1:.4f}")
print(f"ROC-AUC     : {roc_auc:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_odi_venue_test,
        lr_odi_venue_pred,
        target_names=[
            "Team 2 Win",
            "Team 1 Win"
        ],
        zero_division=0
    )
)

# Save results for later comparison
lr_odi_venue_results = {
    "Accuracy": accuracy,
    "Macro F1": macro_f1,
    "Weighted F1": weighted_f1,
    "ROC-AUC": roc_auc
}

print("\n✓ Venue experiment completed.")

ODI — LOGISTIC REGRESSION + VENUE
Accuracy    : 0.6131
Macro F1    : 0.6095
Weighted F1 : 0.6076
ROC-AUC     : 0.6572

Classification Report:
              precision    recall  f1-score   support

  Team 2 Win       0.68      0.49      0.57       193
  Team 1 Win       0.57      0.75      0.65       174

    accuracy                           0.61       367
   macro avg       0.63      0.62      0.61       367
weighted avg       0.63      0.61      0.61       367


✓ Venue experiment completed.


In [ ]:
# ==========================================
# ODI FEATURE EXPERIMENT — CELL 7
# BASELINE + TOSS
# ==========================================

odi_features_toss = odi_features + [
    "toss_winner_diff",
    "bat_first_diff"
]

X_odi_toss = odi_matches[
    odi_features_toss
].copy()

y_odi_toss = odi_matches["target"].copy()

n = len(odi_matches)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_odi_toss_train = X_odi_toss.iloc[:train_end].copy()
X_odi_toss_val = X_odi_toss.iloc[train_end:val_end].copy()
X_odi_toss_test = X_odi_toss.iloc[val_end:].copy()

y_odi_toss_train = y_odi_toss.iloc[:train_end].copy()
y_odi_toss_val = y_odi_toss.iloc[train_end:val_end].copy()
y_odi_toss_test = y_odi_toss.iloc[val_end:].copy()

print("ODI + TOSS DATASET")
print("=" * 50)
print("Features:", len(odi_features_toss))
print("Train:", len(X_odi_toss_train))
print("Validation:", len(X_odi_toss_val))
print("Test:", len(X_odi_toss_test))

print("\nMissing values:")
print(X_odi_toss.isna().sum())

ODI + TOSS DATASET
Features: 9
Train: 1709
Validation: 366
Test: 367

Missing values:
elo_diff               0
win_rate_diff          0
recent_form_diff       0
avg_runs_diff          0
avg_wickets_diff       0
run_rate_diff          0
team_1_h2h_win_rate    0
toss_winner_diff       0
bat_first_diff         0
dtype: int64


In [ ]:
# ==========================================
# ODI FEATURE EXPERIMENT — CELL 8
# LOGISTIC REGRESSION + TOSS
# ==========================================

lr_odi_toss = Pipeline([
    ("scaler", StandardScaler()),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            random_state=42
        )
    )
])

lr_odi_toss.fit(
    X_odi_toss_train,
    y_odi_toss_train
)

lr_odi_toss_pred = lr_odi_toss.predict(
    X_odi_toss_test
)

lr_odi_toss_proba = lr_odi_toss.predict_proba(
    X_odi_toss_test
)

print("ODI Logistic Regression + Toss trained successfully.")

ODI Logistic Regression + Toss trained successfully.


In [ ]:
# ==========================================
# ODI FEATURE EXPERIMENT — CELL 9
# RANDOM FOREST + TOSS
# ==========================================

rf_odi_toss = RandomForestClassifier(
    n_estimators=400,
    max_depth=8,
    min_samples_leaf=4,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_odi_toss.fit(
    X_odi_toss_train,
    y_odi_toss_train
)

rf_odi_toss_pred = rf_odi_toss.predict(
    X_odi_toss_test
)

rf_odi_toss_proba = rf_odi_toss.predict_proba(
    X_odi_toss_test
)

print("ODI Random Forest + Toss trained successfully.")

ODI Random Forest + Toss trained successfully.


In [ ]:
# ==========================================
# ODI FEATURE EXPERIMENT — CELL 10
# EVALUATE LR + TOSS
# ==========================================

accuracy = accuracy_score(
    y_odi_toss_test,
    lr_odi_toss_pred
)

macro_f1 = f1_score(
    y_odi_toss_test,
    lr_odi_toss_pred,
    average="macro"
)

weighted_f1 = f1_score(
    y_odi_toss_test,
    lr_odi_toss_pred,
    average="weighted"
)

roc_auc = roc_auc_score(
    y_odi_toss_test,
    lr_odi_toss_proba[:, 1]
)

lr_odi_toss_results = {
    "Accuracy": accuracy,
    "Macro F1": macro_f1,
    "Weighted F1": weighted_f1,
    "ROC-AUC": roc_auc
}

print("=" * 60)
print("ODI — LOGISTIC REGRESSION + TOSS")
print("=" * 60)

print(f"Accuracy    : {accuracy:.4f}")
print(f"Macro F1    : {macro_f1:.4f}")
print(f"Weighted F1 : {weighted_f1:.4f}")
print(f"ROC-AUC     : {roc_auc:.4f}")

ODI — LOGISTIC REGRESSION + TOSS
Accuracy    : 0.6403
Macro F1    : 0.6399
Weighted F1 : 0.6392
ROC-AUC     : 0.6540


In [ ]:
# ==========================================
# ODI FEATURE EXPERIMENT — CELL 11
# EVALUATE RF + TOSS
# ==========================================

accuracy = accuracy_score(
    y_odi_toss_test,
    rf_odi_toss_pred
)

macro_f1 = f1_score(
    y_odi_toss_test,
    rf_odi_toss_pred,
    average="macro"
)

weighted_f1 = f1_score(
    y_odi_toss_test,
    rf_odi_toss_pred,
    average="weighted"
)

roc_auc = roc_auc_score(
    y_odi_toss_test,
    rf_odi_toss_proba[:, 1]
)

rf_odi_toss_results = {
    "Accuracy": accuracy,
    "Macro F1": macro_f1,
    "Weighted F1": weighted_f1,
    "ROC-AUC": roc_auc
}

print("=" * 60)
print("ODI — RANDOM FOREST + TOSS")
print("=" * 60)

print(f"Accuracy    : {accuracy:.4f}")
print(f"Macro F1    : {macro_f1:.4f}")
print(f"Weighted F1 : {weighted_f1:.4f}")
print(f"ROC-AUC     : {roc_auc:.4f}")

ODI — RANDOM FOREST + TOSS
Accuracy    : 0.6240
Macro F1    : 0.6232
Weighted F1 : 0.6223
ROC-AUC     : 0.6627


In [ ]:
# ==========================================
# ODI FEATURE EXPERIMENT — CELL 12
# FINAL COMPARISON
# ==========================================

# ------------------------------------------
# 1. ODI BASELINE
# ------------------------------------------

X_odi_base = odi_matches[
    odi_features
].copy()

y_odi_base = odi_matches["target"].copy()

n = len(odi_matches)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_base_test = X_odi_base.iloc[val_end:]
y_base_test = y_odi_base.iloc[val_end:]


# Train baseline Logistic Regression
lr_odi_baseline = Pipeline([
    ("scaler", StandardScaler()),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            random_state=42
        )
    )
])

lr_odi_baseline.fit(
    X_odi_base.iloc[:train_end],
    y_odi_base.iloc[:train_end]
)

base_pred = lr_odi_baseline.predict(
    X_base_test
)

base_proba = lr_odi_baseline.predict_proba(
    X_base_test
)

odi_baseline_results = {
    "Accuracy": accuracy_score(
        y_base_test,
        base_pred
    ),

    "Macro F1": f1_score(
        y_base_test,
        base_pred,
        average="macro"
    ),

    "Weighted F1": f1_score(
        y_base_test,
        base_pred,
        average="weighted"
    ),

    "ROC-AUC": roc_auc_score(
        y_base_test,
        base_proba[:, 1]
    )
}


# ------------------------------------------
# 2. COMPARISON
# ------------------------------------------

comparison_odi = pd.DataFrame(
    [
        odi_baseline_results,
        lr_odi_venue_results,
        lr_odi_toss_results
    ],
    index=[
        "ODI — Baseline",
        "ODI — + Venue",
        "ODI — + Toss"
    ]
)

print("=" * 70)
print("              ODI FEATURE COMPARISON")
print("=" * 70)

display(
    comparison_odi.round(4)
)


# ------------------------------------------
# 3. ACCURACY CHANGE
# ------------------------------------------

baseline_accuracy = (
    odi_baseline_results["Accuracy"]
)

venue_accuracy = (
    lr_odi_venue_results["Accuracy"]
)

toss_accuracy = (
    lr_odi_toss_results["Accuracy"]
)

print("\nACCURACY CHANGE")
print("=" * 50)

print(
    f"Baseline : {baseline_accuracy:.4f}"
)

print(
    f"+ Venue  : {venue_accuracy:.4f} "
    f"({venue_accuracy - baseline_accuracy:+.4f})"
)

print(
    f"+ Toss   : {toss_accuracy:.4f} "
    f"({toss_accuracy - baseline_accuracy:+.4f})"
)

              ODI FEATURE COMPARISON


,Accuracy,Macro F1,Weighted F1,ROC-AUC
ODI — Baseline,0.6158,0.6102,0.6078,0.6553
ODI — + Venue,0.6131,0.6095,0.6076,0.6572
ODI — + Toss,0.6403,0.6399,0.6392,0.6540



ACCURACY CHANGE
Baseline : 0.6158
+ Venue  : 0.6131 (-0.0027)
+ Toss   : 0.6403 (+0.0245)


In [ ]:
# ==========================================
# ODI FEATURE EXPERIMENT — CELL 13
# BASELINE + VENUE + TOSS
# ==========================================

odi_features_venue_toss = odi_features + [
    "venue_win_rate_diff",
    "toss_winner_diff",
    "bat_first_diff"
]

X_odi_venue_toss = odi_matches[
    odi_features_venue_toss
].copy()

y_odi_venue_toss = odi_matches["target"].copy()

n = len(odi_matches)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_odi_vt_train = X_odi_venue_toss.iloc[:train_end]
X_odi_vt_test = X_odi_venue_toss.iloc[val_end:]

y_odi_vt_train = y_odi_venue_toss.iloc[:train_end]
y_odi_vt_test = y_odi_venue_toss.iloc[val_end:]

print("ODI + VENUE + TOSS")
print("=" * 50)

print("Features:", len(odi_features_venue_toss))
print("Train:", len(X_odi_vt_train))
print("Test:", len(X_odi_vt_test))

ODI + VENUE + TOSS
Features: 10
Train: 1709
Test: 367


In [ ]:
# ==========================================
# CELL 14 — LOGISTIC REGRESSION
# ==========================================

lr_odi_vt = Pipeline([
    ("scaler", StandardScaler()),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            random_state=42
        )
    )
])

lr_odi_vt.fit(
    X_odi_vt_train,
    y_odi_vt_train
)

lr_odi_vt_pred = lr_odi_vt.predict(
    X_odi_vt_test
)

lr_odi_vt_proba = lr_odi_vt.predict_proba(
    X_odi_vt_test
)

print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


In [ ]:
# ==========================================
# CELL 15 — EVALUATE
# ==========================================

accuracy = accuracy_score(
    y_odi_vt_test,
    lr_odi_vt_pred
)

macro_f1 = f1_score(
    y_odi_vt_test,
    lr_odi_vt_pred,
    average="macro"
)

weighted_f1 = f1_score(
    y_odi_vt_test,
    lr_odi_vt_pred,
    average="weighted"
)

roc_auc = roc_auc_score(
    y_odi_vt_test,
    lr_odi_vt_proba[:, 1]
)

lr_odi_vt_results = {
    "Accuracy": accuracy,
    "Macro F1": macro_f1,
    "Weighted F1": weighted_f1,
    "ROC-AUC": roc_auc
}

print("=" * 60)
print("ODI — LOGISTIC REGRESSION + VENUE + TOSS")
print("=" * 60)

print(f"Accuracy    : {accuracy:.4f}")
print(f"Macro F1    : {macro_f1:.4f}")
print(f"Weighted F1 : {weighted_f1:.4f}")
print(f"ROC-AUC     : {roc_auc:.4f}")

ODI — LOGISTIC REGRESSION + VENUE + TOSS
Accuracy    : 0.6240
Macro F1    : 0.6235
Weighted F1 : 0.6228
ROC-AUC     : 0.6568


In [ ]:
# ==========================================
# CELL 16 — FINAL ODI FEATURE COMPARISON
# ==========================================

odi_all_results = pd.DataFrame(
    [
        odi_baseline_results,
        lr_odi_venue_results,
        lr_odi_toss_results,
        lr_odi_vt_results
    ],
    index=[
        "ODI — Baseline",
        "ODI — + Venue",
        "ODI — + Toss",
        "ODI — + Venue + Toss"
    ]
)

print("=" * 70)
print("              ODI FEATURE EXPERIMENT")
print("=" * 70)

display(
    odi_all_results.round(4)
)

print("\nACCURACY CHANGES")
print("=" * 50)

base = odi_baseline_results["Accuracy"]

for name, row in odi_all_results.iterrows():

    change = row["Accuracy"] - base

    print(
        f"{name}: "
        f"{row['Accuracy']:.4f} "
        f"({change:+.4f})"
    )

              ODI FEATURE EXPERIMENT


,Accuracy,Macro F1,Weighted F1,ROC-AUC
ODI — Baseline,0.6158,0.6102,0.6078,0.6553
ODI — + Venue,0.6131,0.6095,0.6076,0.6572
ODI — + Toss,0.6403,0.6399,0.6392,0.6540
ODI — + Venue + Toss,0.6240,0.6235,0.6228,0.6568



ACCURACY CHANGES
ODI — Baseline: 0.6158 (+0.0000)
ODI — + Venue: 0.6131 (-0.0027)
ODI — + Toss: 0.6403 (+0.0245)
ODI — + Venue + Toss: 0.6240 (+0.0082)


In [ ]:
# ==========================================
# CELL 17 — RANDOM FOREST + VENUE + TOSS
# ==========================================

rf_odi_vt = RandomForestClassifier(
    n_estimators=400,
    max_depth=8,
    min_samples_leaf=4,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_odi_vt.fit(
    X_odi_vt_train,
    y_odi_vt_train
)

rf_odi_vt_pred = rf_odi_vt.predict(
    X_odi_vt_test
)

rf_odi_vt_proba = rf_odi_vt.predict_proba(
    X_odi_vt_test
)

rf_vt_accuracy = accuracy_score(
    y_odi_vt_test,
    rf_odi_vt_pred
)

rf_vt_macro_f1 = f1_score(
    y_odi_vt_test,
    rf_odi_vt_pred,
    average="macro"
)

rf_vt_weighted_f1 = f1_score(
    y_odi_vt_test,
    rf_odi_vt_pred,
    average="weighted"
)

rf_vt_roc_auc = roc_auc_score(
    y_odi_vt_test,
    rf_odi_vt_proba[:, 1]
)

rf_odi_vt_results = {
    "Accuracy": rf_vt_accuracy,
    "Macro F1": rf_vt_macro_f1,
    "Weighted F1": rf_vt_weighted_f1,
    "ROC-AUC": rf_vt_roc_auc
}

print("=" * 60)
print("ODI — RANDOM FOREST + VENUE + TOSS")
print("=" * 60)

print(f"Accuracy    : {rf_vt_accuracy:.4f}")
print(f"Macro F1    : {rf_vt_macro_f1:.4f}")
print(f"Weighted F1 : {rf_vt_weighted_f1:.4f}")
print(f"ROC-AUC     : {rf_vt_roc_auc:.4f}")

ODI — RANDOM FOREST + VENUE + TOSS
Accuracy    : 0.6458
Macro F1    : 0.6455
Weighted F1 : 0.6449
ROC-AUC     : 0.6613


In [ ]:
# ==========================================
# CELL 18 — ODI FINAL DECISION TABLE
# ==========================================

odi_final_comparison = pd.DataFrame(
    [
        odi_baseline_results,
        lr_odi_toss_results,
        lr_odi_vt_results,
        rf_odi_toss_results,
        rf_odi_vt_results
    ],
    index=[
        "LR — Baseline",
        "LR — + Toss",
        "LR — + Venue + Toss",
        "RF — + Toss",
        "RF — + Venue + Toss"
    ]
)

print("=" * 75)
print("                 ODI FINAL EXPERIMENT")
print("=" * 75)

display(
    odi_final_comparison.round(4)
)

                 ODI FINAL EXPERIMENT


,Accuracy,Macro F1,Weighted F1,ROC-AUC
LR — Baseline,0.6158,0.6102,0.6078,0.6553
LR — + Toss,0.6403,0.6399,0.6392,0.6540
LR — + Venue + Toss,0.6240,0.6235,0.6228,0.6568
RF — + Toss,0.6240,0.6232,0.6223,0.6627
RF — + Venue + Toss,0.6458,0.6455,0.6449,0.6613


In [ ]:
# ==========================================
# ODI FINAL MODEL — CELL 19
# ==========================================

# Final feature set
odi_final_features = [
    "elo_diff",
    "win_rate_diff",
    "recent_form_diff",
    "avg_runs_diff",
    "avg_wickets_diff",
    "run_rate_diff",
    "team_1_h2h_win_rate",
    "venue_win_rate_diff",
    "toss_winner_diff",
    "bat_first_diff"
]

X_odi_final = odi_matches[
    odi_final_features
].copy()

y_odi_final = odi_matches["target"].copy()

# Chronological split
n = len(odi_matches)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train_final = X_odi_final.iloc[:val_end]
y_train_final = y_odi_final.iloc[:val_end]

X_test_final = X_odi_final.iloc[val_end:]
y_test_final = y_odi_final.iloc[val_end:]

# Final Random Forest
odi_final_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=8,
    min_samples_leaf=4,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

odi_final_model.fit(
    X_train_final,
    y_train_final
)

print("ODI FINAL MODEL TRAINED")
print("=" * 50)
print("Model: Random Forest")
print("Features:", len(odi_final_features))

for i, feature in enumerate(odi_final_features, 1):
    print(f"{i}. {feature}")

ODI FINAL MODEL TRAINED
Model: Random Forest
Features: 10
1. elo_diff
2. win_rate_diff
3. recent_form_diff
4. avg_runs_diff
5. avg_wickets_diff
6. run_rate_diff
7. team_1_h2h_win_rate
8. venue_win_rate_diff
9. toss_winner_diff
10. bat_first_diff


In [ ]:
# ==========================================
# CELL 20 — FINAL ODI EVALUATION
# ==========================================

final_pred = odi_final_model.predict(
    X_test_final
)

final_proba = odi_final_model.predict_proba(
    X_test_final
)

final_accuracy = accuracy_score(
    y_test_final,
    final_pred
)

final_macro_f1 = f1_score(
    y_test_final,
    final_pred,
    average="macro"
)

final_weighted_f1 = f1_score(
    y_test_final,
    final_pred,
    average="weighted"
)

final_roc_auc = roc_auc_score(
    y_test_final,
    final_proba[:, 1]
)

print("=" * 60)
print("             FINAL ODI MODEL")
print("=" * 60)

print(f"Accuracy    : {final_accuracy:.4f}")
print(f"Macro F1    : {final_macro_f1:.4f}")
print(f"Weighted F1 : {final_weighted_f1:.4f}")
print(f"ROC-AUC     : {final_roc_auc:.4f}")

             FINAL ODI MODEL
Accuracy    : 0.6349
Macro F1    : 0.6339
Weighted F1 : 0.6329
ROC-AUC     : 0.6628


In [ ]:
# ==========================================
# CELL 21 — ODI FEATURE IMPORTANCE
# ==========================================

importance_df = pd.DataFrame({
    "Feature": odi_final_features,
    "Importance": odi_final_model.feature_importances_
})

importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

print("=" * 60)
print("          ODI FEATURE IMPORTANCE")
print("=" * 60)

display(
    importance_df.round(4)
)

importance_df.to_csv(
    "/content/odi_feature_importance.csv",
    index=False
)

          ODI FEATURE IMPORTANCE


,Feature,Importance
0,run_rate_diff,0.1853
1,elo_diff,0.1744
2,avg_runs_diff,0.1667
3,avg_wickets_diff,0.1320
4,win_rate_diff,0.1200
5,team_1_h2h_win_rate,0.0839
6,venue_win_rate_diff,0.0657
7,recent_form_diff,0.0407
8,bat_first_diff,0.0215
9,toss_winner_diff,0.0098


In [ ]:
# ==========================================
# CELL 22 — SAVE ODI MODEL
# ==========================================

import joblib
import json

joblib.dump(
    odi_final_model,
    "/content/odi_match_outcome_model.pkl"
)

joblib.dump(
    odi_final_features,
    "/content/odi_model_features.pkl"
)

odi_model_info = {
    "format": "ODI",
    "model": "Random Forest",
    "accuracy": float(final_accuracy),
    "macro_f1": float(final_macro_f1),
    "weighted_f1": float(final_weighted_f1),
    "roc_auc": float(final_roc_auc),

    "features": odi_final_features,

    "venue_used": True,
    "toss_used": True,
    "pitch_type_used": False,

    "pitch_type_status": "Context/UI only",

    "historical_matches": int(len(odi_matches)),

    "notes": [
        "Venue retained as a required prediction input.",
        "Toss features retained.",
        "Pitch type is currently not used by the ML model.",
        "Final model selected from the tested ODI configurations."
    ]
}

with open(
    "/content/odi_model_info.json",
    "w"
) as f:
    json.dump(
        odi_model_info,
        f,
        indent=4
    )

print("ODI model files saved successfully.")

ODI model files saved successfully.


In [ ]:
# ==========================================
# CELL 23 — SAVE ODI PREDICTION DATA
# ==========================================

odi_matches.to_csv(
    "/content/odi_match_data_for_prediction.csv",
    index=False
)

# Venue history
odi_venue_history_rows = []

for (team, venue), stats in odi_venue_history.items():

    odi_venue_history_rows.append({
        "team": team,
        "venue": venue,
        "matches": stats["matches"],
        "wins": stats["wins"]
    })

pd.DataFrame(
    odi_venue_history_rows
).to_csv(
    "/content/odi_venue_stats.csv",
    index=False
)

print("ODI prediction data saved.")

ODI prediction data saved.


In [ ]:
# ==========================================
# CELL 24 — ODI FINAL FILE CHECK
# ==========================================

import os

odi_files = [
    "odi_match_outcome_model.pkl",
    "odi_model_features.pkl",
    "odi_model_info.json",
    "odi_match_data_for_prediction.csv",
    "odi_venue_stats.csv",
    "odi_feature_importance.csv"
]

print("=" * 60)
print("           ODI FINAL FILE CHECK")
print("=" * 60)

all_ready = True

for filename in odi_files:

    path = "/content/" + filename

    if os.path.exists(path):

        size = os.path.getsize(path) / 1024

        print(f"✓ {filename} ({size:.1f} KB)")

    else:

        print(f"✗ MISSING: {filename}")
        all_ready = False

print()

if all_ready:
    print("✓ ODI MODEL PACKAGE IS READY")
else:
    print("⚠ Some files are missing.")

           ODI FINAL FILE CHECK
✓ odi_match_outcome_model.pkl (7494.8 KB)
✓ odi_model_features.pkl (0.2 KB)
✓ odi_model_info.json (0.9 KB)
✓ odi_match_data_for_prediction.csv (1219.0 KB)
✓ odi_venue_stats.csv (61.9 KB)
✓ odi_feature_importance.csv (0.4 KB)

✓ ODI MODEL PACKAGE IS READY


In [ ]:
# ==========================================
# CELL 24 — ODI FINAL FILE CHECK
# ==========================================

import os

odi_files = [
    "odi_match_outcome_model.pkl",
    "odi_model_features.pkl",
    "odi_model_info.json",
    "odi_match_data_for_prediction.csv",
    "odi_venue_stats.csv",
    "odi_feature_importance.csv"
]

print("=" * 60)
print("           ODI FINAL FILE CHECK")
print("=" * 60)

all_ready = True

for filename in odi_files:

    path = "/content/" + filename

    if os.path.exists(path):

        size = os.path.getsize(path) / 1024

        print(f"✓ {filename} ({size:.1f} KB)")

    else:

        print(f"✗ MISSING: {filename}")
        all_ready = False

print()

if all_ready:
    print("✓ ODI MODEL PACKAGE IS READY")
else:
    print("⚠ Some files are missing.")

           ODI FINAL FILE CHECK
✓ odi_match_outcome_model.pkl (7494.8 KB)
✓ odi_model_features.pkl (0.2 KB)
✓ odi_model_info.json (0.9 KB)
✓ odi_match_data_for_prediction.csv (1219.0 KB)
✓ odi_venue_stats.csv (61.9 KB)
✓ odi_feature_importance.csv (0.4 KB)

✓ ODI MODEL PACKAGE IS READY


In [ ]:
# ==========================================
# CELL 26 — SAVE ODI MODEL
# ==========================================

joblib.dump(
    odi_final_model,
    "/content/odi_match_outcome_model.pkl"
)

joblib.dump(
    odi_final_features,
    "/content/odi_model_features.pkl"
)

print("✓ odi_match_outcome_model.pkl")
print("✓ odi_model_features.pkl")

✓ odi_match_outcome_model.pkl
✓ odi_model_features.pkl


In [ ]:
# ==========================================
# CELL 27 — PREDICTION DATABASE
# ==========================================

odi_matches.to_csv(
    "/content/odi_match_data_for_prediction.csv",
    index=False
)

# Save feature importance
importance_df.to_csv(
    "/content/odi_feature_importance.csv",
    index=False
)

print("✓ ODI prediction database saved")
print("✓ ODI feature importance saved")

✓ ODI prediction database saved
✓ ODI feature importance saved


In [ ]:
# ==========================================
# CELL 28 — FILE VERIFICATION
# ==========================================

import os

odi_final_files = [
    "odi_match_outcome_model.pkl",
    "odi_model_features.pkl",
    "odi_model_info.json",
    "odi_match_data_for_prediction.csv",
    "odi_feature_importance.csv",
    "odi_venue_stats.csv"
]

print("=" * 60)
print("ODI FINAL FILE CHECK")
print("=" * 60)

ready = True

for filename in odi_final_files:

    path = "/content/" + filename

    if os.path.exists(path):
        size = os.path.getsize(path) / 1024
        print(f"✓ {filename} — {size:.1f} KB")
    else:
        print(f"✗ MISSING — {filename}")
        ready = False

print()

if ready:
    print("✓ ODI PACKAGE READY")
else:
    print("⚠ CHECK MISSING FILES")

ODI FINAL FILE CHECK
✓ odi_match_outcome_model.pkl — 7494.8 KB
✓ odi_model_features.pkl — 0.2 KB
✓ odi_model_info.json — 0.9 KB
✓ odi_match_data_for_prediction.csv — 1219.0 KB
✓ odi_feature_importance.csv — 0.4 KB
✓ odi_venue_stats.csv — 61.9 KB

✓ ODI PACKAGE READY


In [ ]:
# ==========================================
# CELL 29 — CREATE ODI ZIP
# ==========================================

import zipfile

odi_zip_path = "/content/MATCHAI_ODI_Final_Model.zip"

with zipfile.ZipFile(
    odi_zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as zipf:

    for filename in odi_final_files:

        filepath = "/content/" + filename

        if os.path.exists(filepath):
            zipf.write(
                filepath,
                arcname=filename
            )

print("=" * 60)
print("MATCHAI ODI FINAL PACKAGE")
print("=" * 60)

with zipfile.ZipFile(
    odi_zip_path,
    "r"
) as zipf:

    for name in zipf.namelist():
        print("✓", name)

print("\nZIP created successfully.")

MATCHAI ODI FINAL PACKAGE
✓ odi_match_outcome_model.pkl
✓ odi_model_features.pkl
✓ odi_model_info.json
✓ odi_match_data_for_prediction.csv
✓ odi_feature_importance.csv
✓ odi_venue_stats.csv

ZIP created successfully.


In [ ]:
# ==========================================
# CELL 30 — DOWNLOAD ODI MODEL
# ==========================================

from google.colab import files

files.download(
    "/content/MATCHAI_ODI_Final_Model.zip"
)

print("✓ ODI model downloaded")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ ODI model downloaded
